# Benchmark MGGP — Biblioteca Python

**Objetivo:** Coleta 10 modelos MIMO aceitos (FreeRun RMSE ∈ (0, 100)) no dataset Wang bicicleta.  
**Saída:** `resultados/resultados_python.csv` + modelos `.pkl` + curvas de convergência `.csv`

---
**Fluxo:**
1. Configure os parâmetros na Célula 1
2. Execute as células em ordem (Shift+Enter)
3. A fábrica roda automaticamente até 10 modelos aceitos (máx. 50 tentativas)
4. Pode interromper (Kernel → Interrupt) e retomar — os modelos já salvos são preservados

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 1 — CONFIGURAÇÃO  (edite aqui antes de rodar)
# ═══════════════════════════════════════════════════════════════

import os

DB_DIR = r"D:\ProjetosPessoais\IC\Database"   # pasta com os Excel Wang

# Parâmetros fixos da competição
PARAMS = dict(
    nDelays          = [1, 2, 5, 10, 25, 50],
    generations      = 300,
    populationSize   = 300,
    evaluationMode   = "RMSE",
    k                = 300,
    evaluationType   = "MShooting",
    evaluationTypeTest="FreeRun",
    nTerms           = 7,
    maxHeight        = 7,
    mutationRate     = 0.3,
    crossoverRate    = 0.8,
    elitePercentage  = 10,
    mode             = "MIMO",
    froe_mode        = False,
    operators        = ['add', 'subtraction', 'mul'],
    n_jobs           = -1,   # -1 = todos os núcleos disponíveis
)

MAX_TENTATIVAS = 50
META_MODELOS   = 10
RMSE_MIN, RMSE_MAX = 0.0, 100.0

# Colunas Excel (0-indexed, conforme notebook original)
U_COLS = [14, 2, 22, 23, 11]
Y_COLS = [18, 17]   # [Vx, Vy]

print(f"CPU disponíveis: {os.cpu_count()}")
print(f"n_jobs={PARAMS['n_jobs']} → processos paralelos: {os.cpu_count() if PARAMS['n_jobs']==-1 else PARAMS['n_jobs']}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 2 — IMPORTS E SETUP
# ═══════════════════════════════════════════════════════════════

import sys, time
import numpy as np
import pandas as pd
import dill

_HERE       = os.path.dirname(os.path.abspath("__file__"))
RESULTS_DIR = os.path.join(_HERE, "resultados")
os.makedirs(RESULTS_DIR, exist_ok=True)

sys.path.insert(0, _HERE)
from src.mggp import MGGP

TRACKS = [
    ("wang21", os.path.join(DB_DIR, "wang21dv_bic_MGGP.xlsx")),
    ("wang22", os.path.join(DB_DIR, "wang22dv_bic_MGGP.xlsx")),
    ("wang31", os.path.join(DB_DIR, "wang31dv_bic_MGGP.xlsx")),
    ("wang81", os.path.join(DB_DIR, "wang81dv_bic_MGGP.xlsx")),
]

def load_track(path):
    df = pd.read_excel(path, header=0)
    return (df.iloc[:, U_COLS].values.astype(float),
            df.iloc[:, Y_COLS].values.astype(float))

def freerun_rmse(model, y, u):
    yp, yd = model.predict("FreeRun", y, u)
    return (float(np.sqrt(np.mean((yd[:, 0] - yp[:, 0]) ** 2))),
            float(np.sqrt(np.mean((yd[:, 1] - yp[:, 1]) ** 2))))

def extract_monitoring(mggp, model, tempo_s):
    lb = mggp._logbook
    min_per_gen = lb.chapters['fitness'].select('min')
    avg_per_gen = lb.chapters['fitness'].select('avg')
    gens  = lb.select('gen')
    evals = lb.select('evals')
    fitness_inicial = float(min_per_gen[0]) if min_per_gen else float('nan')
    fitness_final   = float(min_per_gen[-1]) if min_per_gen else float('nan')
    n_avaliacoes    = int(sum(evals))
    best = fitness_inicial
    geracao_melhor = gens[0] if gens else 1
    for g, f in zip(gens, min_per_gen):
        if np.isfinite(f) and f < best * 0.99:
            best = f; geracao_melhor = int(g)
    taxa = (fitness_inicial - fitness_final) / fitness_inicial * 100 \
           if np.isfinite(fitness_inicial) and fitness_inicial > 0 else float('nan')
    n_termos = int(sum(len(output) for output in model))
    max_lag  = int(model.lagMax) if hasattr(model, 'lagMax') else -1
    conv_df = pd.DataFrame({'geracao': gens, 'melhor_fitness': min_per_gen, 'fitness_medio': avg_per_gen})
    return dict(
        tempo_treino_s=round(tempo_s, 2), n_avaliacoes=n_avaliacoes,
        fitness_inicial=round(fitness_inicial, 6), fitness_final=round(fitness_final, 6),
        geracao_melhor=geracao_melhor, taxa_melhoria_pct=round(taxa, 2),
        n_termos_modelo=n_termos, max_lag_modelo=max_lag,
    ), conv_df

print("Setup OK — imports carregados.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 3 — CARREGA DADOS
# ═══════════════════════════════════════════════════════════════

print("Carregando tracks...")
data = {name: load_track(path) for name, path in TRACKS}
u_train, y_train = data["wang21"]
print(f"Treino (Wang 2.1): {y_train.shape[0]} amostras, {u_train.shape[1]} entradas, {y_train.shape[1]} saídas")
for name, (u, y) in data.items():
    print(f"  {name}: {y.shape[0]} amostras")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 4 — FÁBRICA DE MODELOS
# Execute esta célula. Pode interromper e re-executar.
# ═══════════════════════════════════════════════════════════════

results   = []
n_aceitos = 0

for tentativa in range(1, MAX_TENTATIVAS + 1):
    if n_aceitos >= META_MODELOS:
        break

    print(f"\n{'─'*60}")
    print(f"Tentativa {tentativa}  |  aceitos: {n_aceitos}/{META_MODELOS}")
    print(f"{'─'*60}")
    t0 = time.time()
    try:
        mggp = MGGP(inputs=u_train, outputs=y_train, **PARAMS)
        mggp.run()
        tempo_s = time.time() - t0

        model = mggp._hof[0]
        mggp.element.compileModel(model)

        rv, ry = freerun_rmse(model, y_train, u_train)
        rmse_train = (rv + ry) / 2.0
        print(f"FreeRun treino → Vx={rv:.4f}  Vy={ry:.4f}  mean={rmse_train:.4f}  t={tempo_s:.0f}s")

        if not (RMSE_MIN < rmse_train < RMSE_MAX):
            print("  ✗ Rejeitado")
            continue

        n_aceitos += 1
        mon, conv_df = extract_monitoring(mggp, model, tempo_s)

        row = {
            "library": "python", "model_id": n_aceitos,
            "wang21_vx": rv, "wang21_vy": ry,
            "wang22_vx": float('nan'), "wang22_vy": float('nan'),
            "wang31_vx": float('nan'), "wang31_vy": float('nan'),
            "wang81_vx": float('nan'), "wang81_vy": float('nan'),
            **mon,
        }

        for name, (u_t, y_t) in data.items():
            if name == "wang21":
                continue
            rv2, ry2 = freerun_rmse(model, y_t, u_t)
            row[f"{name}_vx"] = rv2
            row[f"{name}_vy"] = ry2
            print(f"  {name}: Vx={rv2:.4f}  Vy={ry2:.4f}")

        results.append(row)
        print(f"  ✓ Modelo {n_aceitos} aceito | gen_melhor={mon['geracao_melhor']} "
              f"| evals={mon['n_avaliacoes']} | melhoria={mon['taxa_melhoria_pct']:.1f}%")

        # Salva imediatamente
        with open(os.path.join(RESULTS_DIR, f"model_py_id{n_aceitos}.pkl"), "wb") as f:
            dill.dump(model, f)
        conv_df.to_csv(os.path.join(RESULTS_DIR, f"conv_py_id{n_aceitos}.csv"), index=False)
        print(f"  Salvo: model_py_id{n_aceitos}.pkl")

    except Exception as e:
        import traceback
        print(f"  ERRO: {e}")
        traceback.print_exc()

print(f"\n{'═'*60}")
print(f"Fábrica concluída: {n_aceitos}/{META_MODELOS} modelos em {tentativa} tentativas")
if tentativa > 0:
    print(f"Taxa de aceitação: {n_aceitos/tentativa*100:.1f}%")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 5 — SALVA CSV E EXIBE RESUMO
# ═══════════════════════════════════════════════════════════════

if not results:
    print("Nenhum modelo aceito — nada para salvar.")
else:
    df_out = pd.DataFrame(results)
    num_cols = [c for c in df_out.columns if c not in ('library', 'model_id')]
    df_mean = {'library': 'python_MEAN', 'model_id': 0}
    df_mean.update({c: df_out[c].mean() for c in num_cols})
    df_std  = {'library': 'python_STD',  'model_id': 0}
    df_std.update({c: df_out[c].std() for c in num_cols})
    df_out = pd.concat([df_out, pd.DataFrame([df_mean]), pd.DataFrame([df_std])], ignore_index=True)

    csv_path = os.path.join(RESULTS_DIR, "resultados_python.csv")
    df_out.to_csv(csv_path, index=False)
    print(f"CSV salvo em: {csv_path}")
    display(df_out)